# Run the full LIFE method on ISOT (human fake-vs-real) — negative control

Runs the **actual LIFE pipeline** (not a baseline) end-to-end on the **ISOT** dataset, which is
**human-written on both sides** (Fake.csv / True.csv). This is a bigger, independent replication
of the HF-vs-HR negative control (MEMORY §7j): if LIFE's LLM-perplexity fingerprint carries no
veracity signal for human news, accuracy here should sit near **50% (chance)**.

**Labels:** Fake → `human_fake`, True → `human_true` (LIFE 4-class ids 4 / 9). Same feature shape
and model you already ran for MF-vs-MR — only the input data and `--label_set human` change.

**⚠️ Read the result carefully.** ISOT is **not source-matched**: ~99% of True.csv is Reuters
newswire, ~0% of Fake.csv is. So "fake vs real" is confounded with "tabloid vs Reuters style".
→ A **near-chance** result cleanly supports *"LIFE can't recognize human-written fake news"*; an
**above-chance** result is **ambiguous** (GPT-2 perplexity may be reading newswire style, not
veracity). The clean, topic-matched control remains HF-vs-HR.

**Scale:** 1,000 articles/class (LIFE-benchmark scale). Full ISOT (~45k) would be many GPU-hours.
**GPU required** (Runtime → Change runtime type → GPU) — stages 1 (BERT) and 3 (GPT-2) need it.

**5 stages:** 0 convert+subsample → 1 key-sentence extraction (BERT) → 2 merge sentences →
3 GPT-2 perplexity features → 4 split + train the LIFE classifier.

In [ ]:
!pip install -q transformers datasets nltk tqdm  # torch is preinstalled on Colab

In [ ]:
import torch, nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_DIR = '/content/drive/MyDrive/LIFE'
ISOT_DIR    = f'{PROJECT_DIR}/dataset/data/ISOT'          # holds Fake.csv / True.csv
WORK        = f'{ISOT_DIR}/isot_life'                     # all pipeline artifacts live here
CONVERTED   = f'{WORK}/converted'                         # stage 0 out: ISOT_fake/true.jsonl
FEATURES    = f'{WORK}/features'                          # stage 3 out: GPT-2 feature jsonl
KEYSENTS    = f'{WORK}/keysents.jsonl'                    # stage 1 out
BERT_CKPT   = f'{WORK}/isot_bert_model.pt'                # stage 1 key-sentence extractor
TRAIN_JSONL = f'{WORK}/train.jsonl'                       # stage 4 split (kept OUT of FEATURES
TEST_JSONL  = f'{WORK}/test.jsonl'                        #   so it is not re-ingested as input)

os.chdir(PROJECT_DIR)  # so `dataset/...` and `LIFE_train/...` script paths resolve
os.makedirs(WORK, exist_ok=True)
print('ISOT Fake.csv found:', os.path.isfile(f'{ISOT_DIR}/Fake.csv'))
print('ISOT True.csv found:', os.path.isfile(f'{ISOT_DIR}/True.csv'))

## Stage 0 — convert ISOT CSV → JSONL, subsample 1,000 / class
Fake → `human_fake`, True → `human_true`; synthesizes ids (ISOT has none).

In [ ]:
!python dataset/0_convert_isot.py --input_dir "{ISOT_DIR}" --output_dir "{CONVERTED}" --sample_size 1000 --seed 42

## Stage 1 — key-sentence extraction (BERT)
Trains `bert-base-uncased` to separate fake/true, then leave-one-sentence-out to pick the top-10
sentences per article. **The slow stage** (~10–20 min on a T4 for 2k articles). *Note: on ISOT the
BERT will separate the two classes easily via the Reuters artifact — but it only **selects which
sentences get GPT-2-scored**; LIFE's verdict comes from the perplexity features in stage 4.*

In [ ]:
!python dataset/1_keySentenceExtraction.py --data_dir "{CONVERTED}" --output_file "{KEYSENTS}" --top_k 10 --model_path "{BERT_CKPT}" --gpu 0

## Stage 2 — merge the key sentences back into the article JSONL
In-place: adds a `sentence` field to `CONVERTED/ISOT_fake.jsonl` and `ISOT_true.jsonl`.

In [ ]:
!python dataset/2_concate.py --folder_path "{CONVERTED}" --important_sentences_file "{KEYSENTS}"

## Stage 3 — GPT-2 perplexity features
Computes the per-token log-likelihood "fingerprint" (the LIFE feature) over each article, reusing
`EN_LABELS` (which already maps `human_fake`→4 / `human_true`→9). ~5–15 min on GPU.

In [ ]:
!python dataset/3_gen_features_local.py --input_dir "{CONVERTED}" --output_dir "{FEATURES}" --model gpt2 --gpu 0

## Stage 4 — split + train the LIFE classifier (human label set)
Trains LIFE's token-level Transformer classifier on the perplexity features, **`--label_set human`**
(HF-vs-HR / ISOT). Fast (small features). **Watch the printed accuracy vs 50%:** near-chance → LIFE
has no fingerprint for human-written fake news (the point of this experiment). `real=human_true`,
`fake=human_fake`.

In [ ]:
!python LIFE_train/train.py --model Transformer --train_mode classify --label_set human --split_dataset --data_path "{FEATURES}" --train_path "{TRAIN_JSONL}" --test_path "{TEST_JSONL}" --train_ratio 0.8 --num_train_epochs 50 --seq_len 1024 --batch_size 32

## Notes
- **What success looks like:** accuracy ≈ 50% and low macro-F1 → confirms LIFE's fingerprint can't
  separate human fake from human real (replicates §7j on an independent, larger, different-domain
  human corpus). An above-chance number is **not** a refutation — see the Reuters-artifact caveat
  in the header.
- **Sync to Drive before running:** `dataset/0_convert_isot.py` (new) and the edited
  `LIFE_train/train.py` (added `--label_set`). Stages 1–3 (`1_keySentenceExtraction.py`,
  `2_concate.py`, `3_gen_features_local.py`) and `backend_utils.py` are unchanged from your MF-vs-MR
  run — make sure they're already on Drive.
- **Change scale:** edit `--sample_size` in Stage 0 (0 = all ~45k). **Re-run cleanly:** delete
  `WORK` first; Stage 1 *loads* `BERT_CKPT` if it exists instead of retraining.
- **Compare against the paper's task:** drop `--label_set human` (defaults to `gpt35`, MF-vs-MR).